In [0]:
# ============================================================
# NOTEBOOK : DIM_PRODUCT
# PURPOSE  : PRODUCT DIMENSION INCREMENTAL LOAD
# ============================================================

# Import required packages

from pyspark.sql.functions import *
from delta.tables import *
import uuid

In [0]:
%run /Users/ragul.p.dev@gmail.com/fabric_incremental_project/Functions/FN_Common_Functions

In [0]:
%run "/Users/ragul.p.dev@gmail.com/fabric_incremental_project/Functions/FN_LOGGER"

In [0]:
# ============================================================
# GET BRONZE DATA
# ============================================================

try:

    metadata = get_metadata("product_tbl")

    table_name = metadata["target_table"]

    source_system = metadata["source_system"]

    watermark_column = metadata["watermark_column"]

    primary_key = metadata["primary_key_column"]

    source_table = f"bronze.{table_name}"

    target_table = "silver.dim_product"

    pipeline_name = "PL_DIM_PRODUCT"

    pipeline_run_id = str(uuid.uuid4())

    start_time = get_current_timestamp()

    last_watermark = get_watermark(table_name)

    print(f"Last Watermark : {last_watermark}")

    bronze_df = spark.sql(f"""

        SELECT

            product_id,
            product_name,
            category_id,
            supplier_id,
            unit_price,
            product_status,
            stock_quantity,
            modified_date

        FROM {source_table}

        WHERE {watermark_column} > '{last_watermark}'

    """)

    bronze_df.createOrReplaceTempView(
        "vw_bronze_product"
    )

    print(f"Bronze View Created : {table_name}")

    rows_read = bronze_df.count()

    print(f"Rows Read : {rows_read}")

except Exception as e:

    print(f"Bronze View Creation Failed : {table_name}")

    raise(e)



Last Watermark : 1900-01-01 00:00:00
Bronze View Created : product_tbl
Rows Read : 5


Max Date updated successfully


FN_COMMON_FUNCTIONS LOADED SUCCESSFULLY


FN_LOGGER LOADED SUCCESSFULLY


In [0]:
# ============================================================
# CREATE SILVER VIEW
# ============================================================

try:

    silver_df = spark.sql("""

        SELECT

            p.product_id,

            p.product_name,

            c.category_name,

            s.supplier_name,

            p.unit_price as Price,

            p.stock_quantity,

            p.product_status,

            CASE

                WHEN p.unit_price >= 30000
                THEN 'PREMIUM'

                WHEN p.unit_price >= 5000
                THEN 'MEDIUM'

                ELSE 'STANDARD'

            END AS product_category,

            CASE

                WHEN p.stock_quantity <= 20
                THEN 'LOW_STOCK'

                ELSE 'AVAILABLE'

            END AS stock_status,

            p.modified_date,

            sha2(
                concat_ws(
                    '|',
                    p.product_name,
                    c.category_name,
                    s.supplier_name,
                    p.unit_price,
                    p.stock_quantity,
                    p.product_status
                ),
                256
            ) AS hash_key,

            current_timestamp()
                AS effective_start_date,

            CAST(NULL AS TIMESTAMP)
                AS effective_end_date,

            1 AS is_current,

            0 AS is_deleted

        FROM vw_bronze_product p

        LEFT JOIN silver.dim_category c
            ON p.category_id = c.category_id
            AND c.is_current = 1

        LEFT JOIN silver.dim_supplier s
            ON p.supplier_id = s.supplier_id
            AND s.is_current = 1

    """)

    silver_df.createOrReplaceTempView(
        "vw_silver_product"
    )

    print(f"Silver View Created : {table_name}")

except Exception as e:

    print(f"Silver View Creation Failed : {table_name}")

    raise(e)

Silver View Created : product_tbl


In [0]:
# CREATE TARGET TABLE
# ============================================================

try:

    spark.sql("""

        CREATE TABLE IF NOT EXISTS silver.dim_product
        (
            product_id BIGINT,
            product_name STRING,
            category_name STRING,
            supplier_name STRING,
            price DOUBLE,
            product_category STRING,
            modified_date TIMESTAMP,
            hash_key STRING,
            effective_start_date TIMESTAMP,
            effective_end_date TIMESTAMP,
            is_current INT,
            is_deleted INT
        )

        USING DELTA

    """)

    print(f"Target Table Created : {target_table}")

except Exception as e:

    print(f"Target Table Creation Failed : {target_table}")

    raise(e)



Target Table Created : silver.dim_product


In [0]:
# ============================================================
# MERGE LOGIC
# ============================================================

try:

    spark.sql("""

        MERGE INTO silver.dim_product AS target

        USING vw_silver_product AS source

        ON target.product_id = source.product_id
           AND target.is_current = 1

        WHEN MATCHED
             AND target.hash_key <> source.hash_key

        THEN UPDATE SET

            target.effective_end_date =
                current_timestamp(),

            target.is_current = 0

        WHEN NOT MATCHED

        THEN INSERT
        (
            product_id,
            product_name,
            category_name,
            supplier_name,
            price,
            product_category,
            modified_date,
            hash_key,
            effective_start_date,
            effective_end_date,
            is_current,
            is_deleted
        )

        VALUES
        (
            source.product_id,
            source.product_name,
            source.category_name,
            source.supplier_name,
            source.price,
            source.product_category,
            source.modified_date,
            source.hash_key,
            source.effective_start_date,
            source.effective_end_date,
            source.is_current,
            source.is_deleted
        )

    """)

    print(f"Merge Completed : {table_name}")

    spark.sql("""

        UPDATE silver.dim_product

        SET

            is_deleted = 1,
            is_current = 0,
            effective_end_date = current_timestamp()

        WHERE product_id NOT IN
        (
            SELECT product_id
            FROM vw_silver_product
        )

        AND is_current = 1

    """)

    print(f"Soft Delete Completed : {table_name}")

except Exception as e:

    print(f"Merge Failed : {table_name}")

    raise(e)



Merge Completed : product_tbl
Soft Delete Completed : product_tbl


In [0]:
# ============================================================
# UPDATE WATERMARK & AUDIT LOG
# ============================================================

try:

    max_date = get_max_date(
        bronze_df,
        watermark_column
    )

    if max_date is not None:

        update_watermark(
            table_name,
            max_date
        )

        print(f"Watermark Updated : {table_name}")

    else:

        print("No Incremental Records Found")

    rows_written = silver_df.count()

    end_time = get_current_timestamp()

    execution_time_seconds = int(
        (end_time - start_time).total_seconds()
    )

    insert_audit_log(

        pipeline_run_id,
        pipeline_name,
        source_system,
        table_name,
        start_time,
        end_time,
        rows_read,
        rows_written,
        "SUCCESS",
        execution_time_seconds

    )

    print(f"DIM_PRODUCT SUCCESSFULLY LOADED : {table_name}")

except Exception as e:

    insert_error_log(

        str(uuid.uuid4()),
        pipeline_run_id,
        table_name,
        source_system,
        "DIM_PRODUCT",
        str(e)

    )

    print(f"DIM_PRODUCT LOAD FAILED : {table_name}")

    raise(e)

Watermark Updated : product_tbl
Watermark Updated : product_tbl
Audit Log Inserted : product_tbl
DIM_PRODUCT SUCCESSFULLY LOADED : product_tbl
